# 02 — Split dan Representasi Teks

Tahap 4. **Belum pernah dijalankan** — mesin penulis kode tidak memegang dataset
maupun model FastText. Jalankan di mesin compute.

Yang dikerjakan:

1. Tiga split terstratifikasi (seed 42/43/44), dibekukan ke `data/processed/`
2. Tokenisasi spaCy, satu pass, lalu FastText dilepas dari memori
3. Dua representasi: sekuens 500×300 untuk CNN1D, dan vektor SIF untuk MLP
4. Reproduksi analisis Fig. 4b paper dengan pasangan salah-eja dari korpus **kita**

## Prasyarat

`pip install -r requirements.txt` saja. Model spaCy dan FastText diunduh
**otomatis** saat pertama dibutuhkan — tidak ada langkah manual.

Model FastText **wajib `.bin`**, bukan `.vec`: hanya `.bin` yang membawa n-gram
subword sehingga kata OOV bisa diberi vektor, dan itu inti argumen paper.
`cc.en.300.bin` berukuran ~7 GB (unduhan ~4,2 GB terkompresi), sekali saja.

**RAM:** model ini butuh ~15 GB saat dimuat. `fasttext.util.reduce_model` tidak
menolong karena harus memuat model 300 dimensi penuh lebih dulu. Notebook ini
mengekstrak fitur satu kali ke disk lalu melepas modelnya — kalau RAM tipis,
biarkan lambat karena swap, jangan potong dimensinya.

**Prasyarat data:** `01_data_audit.ipynb` harus sudah dijalankan sehingga
`data/processed/manifest.csv` ada.

In [ ]:
import sys, time, gc
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import splits as S
from src import text_features as TF
from src.config import CONFIG, set_seed
from src.utils import save_fig

set_seed(42)

manifest = pd.read_csv(CONFIG["data_processed"] / "manifest.csv")
labels = manifest["kelas"].tolist()
print(f"{len(manifest)} pasangan, {manifest['kelas'].nunique()} kelas")

## 1. Split

800 dokumen train (validation dipotong dari dalam 800 itu, bukan ditambahkan),
sisanya test, terstratifikasi per kelas, tiga seed.

**Ini deviasi, bukan protokol paper.** Paper memakai k-fold cross-validation dan
melatih dengan jumlah epoch tetap tanpa early stopping. Kita memakai tiga random
split dan memotong validation untuk early stopping. Keduanya tercatat di
`vault/10-jurnal/1907.06370-spesifikasi-implementasi.md`.

Split disimpan sebagai indeks ke `manifest.csv` plus fingerprint file itu, supaya
split yang dipakai pada manifest berurutan berbeda gagal keras alih-alih menukar
label diam-diam.

In [ ]:
paths = S.write_splits(labels)
for p in paths:
    print(p.name)

split = S.load_split(42, labels)
print(f"\ntrain {len(split['train'])} | val {len(split['val'])} "
      f"| test {len(split['test'])}")

dist = pd.DataFrame(S.describe(split, labels)).fillna(0).astype(int)
dist["train_persen"] = (dist["train"] / dist["train"].sum() * 100).round(1)
dist["korpus_persen"] = (
    pd.Series(labels).value_counts().reindex(dist.index) / len(labels) * 100
).round(1)
display(dist)
print("Kolom dua terakhir harus berdekatan — itu buktinya stratifikasi jalan.")

## 2. Tokenisasi

Satu pass spaCy untuk seluruh korpus, punctuation dibuang. Hasilnya dipakai oleh
**kedua** representasi, jadi tidak ada dokumen yang ditokenisasi dua kali.

In [ ]:
texts = [Path(p).read_text(encoding="utf-8", errors="replace")
         for p in manifest["path_teks"]]

t0 = time.perf_counter()
tokenize = TF.spacy_tokenizer()
token_lists = TF.tokenize_corpus(texts, tokenize)
t_tok = time.perf_counter() - t0

n_tok = np.array([len(t) for t in token_lists])
print(f"tokenisasi {len(texts)} dokumen dalam {t_tok:.1f}s "
      f"({t_tok / len(texts) * 1000:.0f} ms/dokumen)")
print(f"token per dokumen — median {int(np.median(n_tok))}, "
      f"rata-rata {n_tok.mean():.0f}, maks {n_tok.max()}")
print(f"dokumen tanpa token sama sekali: {(n_tok == 0).sum()} "
      f"({(n_tok == 0).mean():.1%})")
print(f"dokumen > {CONFIG['max_words']} token (akan di-truncate): "
      f"{(n_tok > CONFIG['max_words']).mean():.1%}")
print("\nPaper menyebut rata-rata 136 kata (panjang >=4 karakter) untuk korpus ini.")
print("Angka kita menghitung SEMUA token, jadi wajar lebih tinggi. Pembanding "
      "yang setara:")
n_tok4 = np.array([sum(len(w) >= 4 for w in t) for t in token_lists])
print(f"  rata-rata token >=4 karakter: {n_tok4.mean():.0f}  (paper: 136)")

## 3. Analisis OOV dan reproduksi Fig. 4b

Paper memakai kamus Enchant dan membandingkan GloVe, ELMo, dan FastText. Kita
hanya punya FastText, dan tidak memakai kamus eksternal: kata yang sering diulang
korpus dianggap kosakata rujukan, kata nyaris-unik yang ejaannya hampir sama
dengan salah satunya dianggap salah eja OCR.

Itu heuristik, bukan ground truth — karena itu pasangannya dicetak untuk diperiksa
mata, dan angka OOV kita **sebanding bentuknya, bukan nilainya** dengan angka 26%
di paper.

In [ ]:
rates = TF.oov_rate(token_lists)
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(rates, bins=40, color="#4C78A8")
ax.set_xlabel("proporsi token di luar kosakata rujukan korpus")
ax.set_ylabel("jumlah dokumen")
ax.set_title("Distribusi dokumen menurut proporsi kata non-rujukan")
fig.tight_layout()
save_fig(fig, "04_distribusi_oov")

print(f"rata-rata proporsi non-rujukan: {rates.mean():.1%}")
print(f"median                        : {np.median(rates):.1%}")
print(f"dokumen dengan >90% non-rujukan: {(rates > 0.9).sum()} "
      f"({(rates > 0.9).mean():.1%})")
print("\nPaper: ~26% korpus di luar kamus US English, ~23% di luar GloVe, dengan "
      "ekor panjang berisi puluhan dokumen yang isinya hampir seluruhnya bukan "
      "kata Inggris. Bandingkan BENTUK distribusinya, bukan angkanya.")

In [ ]:
# Pasangan salah-eja yang benar-benar ada di korpus kita.
pairs = TF.find_misspelling_pairs(token_lists, limit=40)
print(f"{len(pairs)} kandidat pasangan ditemukan\n")

embed = TF.fasttext_embedder()
baris = []
for salah, benar, freq in pairs:
    baris.append({
        "salah_eja": salah,
        "dugaan_asli": benar,
        "freq": freq,
        "cosine_fasttext": round(TF.cosine(embed(salah), embed(benar)), 3),
    })
fig4b = pd.DataFrame(baris).sort_values("cosine_fasttext", ascending=False)
display(fig4b)

print(f"median cosine: {fig4b['cosine_fasttext'].median():.3f}")
print("\nPaper Fig. 4b, sebagai pembanding (bukan korpus yang sama):")
print("  specifically/Specificalily  GloVe 0.71  ELMo 0.68  FastText 0.96")
print("  filter/fiilter              GloVe 0.91  ELMo 0.73  FastText 0.96")
print("  alcohol/Aleohol             GloVe 0.40  ELMo 0.69  FastText 0.88")
print("  Largely/Largly              GloVe 0.25  ELMo 0.81  FastText 0.98")
print("\nKolom GloVe dan ELMo TIDAK direproduksi — butuh dua model besar lagi.")
print("Catat sebagai reproduksi sebagian, jangan diklaim setara Fig. 4b penuh.")

In [ ]:
# Periksa klaim paper: tidak ada karakter yang OOV bagi FastText.
karakter = sorted({c for t in token_lists for w in t for c in w})
print(f"{len(karakter)} karakter unik di korpus")
kosong = [c for c in karakter if np.allclose(embed(c), 0)]
print("karakter yang vektornya nol (indikasi OOV bagi FastText):",
      kosong or "tidak ada")
print("\nPaper mengklaim Tesseract tidak menghasilkan karakter OOV bagi FastText.")
print("Baris di atas menguji klaim itu di korpus kita.")

## 4. Representasi 1 — sekuens 500×300 untuk CNN1D

Ditulis sebagai `.npy` memmap `float16`: 3482 × 500 × 300 ≈ 1,0 GB, bukan 2,1 GB
kalau `float32`. Kehilangan presisinya tidak berarti dibanding noise OCR.

Dihitung **sekali** lalu dibaca dari disk tiap epoch — jangan hitung ulang.

In [ ]:
seq_path = CONFIG["data_processed"] / "sequences_500x300_fp16.npy"

t0 = time.perf_counter()
TF.build_sequences(token_lists, embed, seq_path)
t_seq = time.perf_counter() - t0

arr = np.load(seq_path, mmap_mode="r")
print(f"{seq_path.name}  shape={arr.shape}  dtype={arr.dtype}")
print(f"ukuran di disk: {seq_path.stat().st_size / 1e9:.2f} GB")
print(f"waktu: {t_seq:.1f}s ({t_seq / len(token_lists) * 1000:.0f} ms/dokumen)")

# sanity: posisi setelah panjang dokumen harus nol
i = int(np.argmin(n_tok[n_tok > 0]))
print(f"\ncontoh dokumen dengan {n_tok[i]} token: "
      f"padding mulai nol di posisi {n_tok[i]}? "
      f"{bool(np.all(arr[i, n_tok[i]:] == 0))}")

## 5. Representasi 2 — vektor SIF untuk MLP

Rata-rata berbobot `a / (a + p(w))`, lalu komponen utama pertama dibuang.

**Komponen utama dipasang pada baris train saja.** Menghitungnya atas seluruh
korpus adalah kebocoran informasi ke test — halus dan mudah terlewat. Karena itu
satu file SIF dibuat per seed, bukan satu untuk semua.

In [ ]:
probs = TF.word_probabilities([token_lists[i] for i in split["train"]])
raw = TF.weighted_average(token_lists, embed, probs, a=1e-3)
print("vektor mentah:", raw.shape)

for seed in CONFIG["seeds"]:
    sp = S.load_split(seed, labels)
    direction = TF.fit_principal_direction(raw[sp["train"]])
    sif = TF.remove_component(raw, direction)
    out = CONFIG["data_processed"] / f"sif_seed{seed}.npy"
    np.save(out, sif.astype(np.float32))
    sisa = float(np.abs(sif[sp["train"]] @ direction).max())
    print(f"{out.name}  shape={sif.shape}  "
          f"sisa proyeksi di train: {sisa:.2e} (harus ~0)")

print("\nCatatan: probs dihitung dari train seed 42. Kalau mau benar-benar bersih "
      "per seed, hitung probs per split juga — dampaknya kecil karena p(w) "
      "didominasi stopword, tapi catat sebagai penyederhanaan.")

In [ ]:
# Lepaskan FastText dari memori sebelum training. Ini bukan kosmetik:
# modelnya ~15 GB dan tidak dibutuhkan lagi setelah fitur ada di disk.
del embed
gc.collect()
print("model FastText dilepas. Fitur sudah ada di disk:")
for f in sorted(CONFIG["data_processed"].glob("*.npy")):
    print(f"  {f.name}  {f.stat().st_size / 1e6:.0f} MB")

## 6. Ringkasan untuk catatan eksperimen

Salin keluaran sel ini ke `vault/30-eksperimen/01-ekstraksi-fitur-teks.md`, lalu
ubah statusnya dari `belum-dijalankan` menjadi `selesai`.

In [ ]:
print(f"""
n_dokumen:: {len(manifest)}
n_train:: {len(split['train'])}
n_val:: {len(split['val'])}
n_test:: {len(split['test'])}
durasi_tokenisasi:: {t_tok:.1f}s
durasi_sekuens:: {t_seq:.1f}s
median_token:: {int(np.median(n_tok))}
rata_token_min4char:: {n_tok4.mean():.0f}
persen_dokumen_kosong:: {(n_tok == 0).mean():.1%}
persen_di_atas_500:: {(n_tok > CONFIG['max_words']).mean():.1%}
rata_proporsi_oov:: {rates.mean():.1%}
median_cosine_salah_eja:: {fig4b['cosine_fasttext'].median():.3f}
ukuran_sekuens_gb:: {seq_path.stat().st_size / 1e9:.2f}
""")